In [ ]:
import os, csv, math
import numpy as np
import pandas as pd
from scipy.signal import butter, lfilter, spectrogram
from collections import Counter
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler

np.random.seed(42)
SR, WIN, STEP = 200, 200, 100
DATA_ROOT = "./EMG_data_for_gestures-master/EMG_data_for_gestures-master"
OUT_CSV = "./data_preprocessed.csv"

def process(sig):
    env = np.abs(sig)
    b, a = butter(4, 5/(0.5*SR), btype='low')
    env_lp = lfilter(b, a, env)
    return (env_lp - env_lp.mean()) / env_lp.std()

def detect_activity(proc):
    summed = proc.sum(axis=1)
    f, t, Sxx = spectrogram(summed, fs=SR, nperseg=WIN, noverlap=WIN//2)
    l2 = np.linalg.norm(Sxx, axis=0)
    thr = np.median(l2)
    idx = np.where(l2 >= thr)[0]
    if len(idx) < 2:
        return 0, len(proc)
    start = int(t[idx[0]] * SR)
    end   = int((t[idx[-1]] + 1e-6) * SR)
    return start, end

def load_file(fp):
    df = pd.read_csv(fp, delim_whitespace=True, skiprows=1, header=None).dropna()
    raw = df.iloc[:,1:9].astype(float).values
    lbl = df.iloc[:,9].astype(int).values
    proc = np.column_stack([process(raw[:,i]) for i in range(8)])
    return proc, lbl

def extract_feats(seg):
    feats = []
    for c in range(seg.shape[1]):
        s = seg[:,c]
        feats.extend([s.mean(), np.sqrt((s**2).mean()), s.var()])
    return feats

X_all, y_all = [], []
for subj in sorted(os.listdir(DATA_ROOT)):
    path = os.path.join(DATA_ROOT, subj)
    if os.path.isdir(path):
        for fname in sorted(os.listdir(path)):
            if fname.endswith(".txt"):
                proc, lbl = load_file(os.path.join(path, fname))
                start, end = detect_activity(proc)
                for i in range(start, end - WIN, STEP):
                    win = proc[i:i+WIN]
                    mid = lbl[i+WIN//2]
                    X_all.append(extract_feats(win))
                    y_all.append(mid)

X_all = np.array(X_all)
y_all = np.array(y_all)

counts = Counter(y_all)
gest_counts = [counts[c] for c in counts if c != 0]
target = int(np.median(gest_counts))

rus0 = RandomUnderSampler(sampling_strategy={0: target}, random_state=42)
X_tmp, y_tmp = rus0.fit_resample(X_all, y_all)

counts_tmp = Counter(y_tmp)
unders = {c: target for c, cnt in counts_tmp.items() if cnt > target}
if unders:
    rus1 = RandomUnderSampler(sampling_strategy=unders, random_state=42)
    X_tmp, y_tmp = rus1.fit_resample(X_tmp, y_tmp)

counts_tmp = Counter(y_tmp)
overs = {c: target for c, cnt in counts_tmp.items() if cnt < target}
if overs:
    ros = RandomOverSampler(sampling_strategy=overs, random_state=42)
    X_bal, y_bal = ros.fit_resample(X_tmp, y_tmp)
else:
    X_bal, y_bal = X_tmp, y_tmp

os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
with open(OUT_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    for feats, lbl in zip(X_bal, y_bal):
        writer.writerow(list(feats) + [lbl])

print("Before balancing:", Counter(y_all))
print("After balancing: ", Counter(y_bal))
print(f"Saved {len(X_bal)} windows to {OUT_CSV}")


C:\Users\user\AppData\Local\Temp\ipykernel_11144\1102397340.py:33: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(fp, delim_whitespace=True, skiprows=1, header=None).dropna()
C:\Users\user\AppData\Local\Temp\ipykernel_11144\1102397340.py:33: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(fp, delim_whitespace=True, skiprows=1, header=None).dropna()
C:\Users\user\AppData\Local\Temp\ipykernel_11144\1102397340.py:33: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(fp, delim_whitespace=True, skiprows=1, header=None).dropna()
C:\Users\user\AppData\Local\Temp\ipykernel_11144\1102397340.py:33: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is de

Before balancing: Counter({np.int64(0): 25108, np.int64(5): 2526, np.int64(4): 2518, np.int64(6): 2515, np.int64(3): 2498, np.int64(2): 2426, np.int64(1): 1434, np.int64(7): 137})
After balancing:  Counter({np.int64(0): 2498, np.int64(1): 2498, np.int64(2): 2498, np.int64(3): 2498, np.int64(4): 2498, np.int64(5): 2498, np.int64(6): 2498, np.int64(7): 2498})
Saved 19984 windows to ./data_preprocessed.csv
